# CS 259 - Hybrid CPU-GPU benchmark on a Colab GPU runtime
Runs the serial, OpenMP, CUDA and hybrid implementations **on the same Colab VM** and saves raw CSV results.

**Before you start**
1. Runtime -> Change runtime type -> **GPU** (any GPU Colab gives you; it is recorded in every result row).
2. Drag `cs259_bundle.zip` (from `build_bundle/` in the project) into the **Files** panel on the left and wait until the upload finishes.
3. Run the cells from top to bottom. Keep this browser tab open and in the foreground while cell 4 runs.

In [ ]:
# 1. Hardware check - the runtime must have an NVIDIA GPU and nvcc
import subprocess, os
def sh(cmd): return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
print(sh("nvidia-smi"))
print(sh("nvcc --version | tail -2"))
print("logical CPUs:", os.cpu_count()); print(sh("lscpu | grep -E 'Model name|Thread|Core|Socket'"))
assert "NVIDIA" in sh("nvidia-smi"), "No GPU: Runtime -> Change runtime type -> GPU"
assert sh("which nvcc"), "nvcc not found in this runtime"

In [ ]:
# 2. Unpack the bundle
assert os.path.exists("/content/cs259_bundle.zip"), "Upload cs259_bundle.zip to the Files panel first"
!rm -rf /content/cs259 && mkdir -p /content/cs259 && unzip -q -o /content/cs259_bundle.zip -d /content/cs259
%cd /content/cs259
!ls

In [ ]:
# 3. Build for this GPU and run the correctness gate (fast; stop here if anything fails)
gpu = sh("nvidia-smi --query-gpu=name --format=csv,noheader").splitlines()[0]
platform = "colab_" + "".join(c if c.isalnum() else "_" for c in gpu).strip("_")
print("platform id:", platform)
!make gpu 2>&1 | tail -25
!rm -f /content/validate_smoke.csv
!build/gpu/nn_validate --platform {platform}_smoke --out /content/validate_smoke.csv | tail -40

In [ ]:
# 4. Full benchmark grid (serial, OpenMP, CUDA with 3 block sizes, hybrid-sequential, hybrid-pipelined)
n = os.cpu_count(); threads = []; t = 1
while t < n: threads.append(t); t *= 2
threads.append(n)
os.environ["THREADS"] = ",".join(map(str, threads)); os.environ["RUNS"] = "20"
print("OpenMP thread counts:", os.environ["THREADS"])
!benchmarks/run_all.sh {platform} gpu 2>&1 | tail -150

In [ ]:
# 5. Collect results and download them
!cd /content/cs259 && zip -r -q /content/results_{platform}.zip results/raw environment && ls -la /content/results_{platform}.zip
from google.colab import files
files.download(f"/content/results_{platform}.zip")   # if the browser blocks this, right-click the zip in the Files panel -> Download